# Welcome to QMC.jl

Original QMCPy demo: [`QMCPy/demos/qmcpy_intro.ipynb`](../../QMCPy/demos/qmcpy_intro.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMC.jl/blob/develop/demos/qmc.jl_intro.ipynb)

This tutorial introduces QMC.jl by walking through the four building blocks:
  1. Discrete Distribution — generates points in [0,1)^d
  2. True Measure — transforms points to the target domain
  3. Integrand — the function to integrate
  4. Stopping Criterion — decides when enough samples have been taken

In [1]:
using QMC
using Statistics
using LinearAlgebra
using Printf

## Importing QMC.jl

Here we show the Julia analogue of the QMCPy introduction.

First, we load the package `QMC.jl` and then explore its main components.

In [2]:
println("="^60)
println("QMC.jl — Julia package for Quasi-Monte Carlo integration")
println("="^60)
println("  `using QMC` imports all types and functions.")
println()

QMC.jl — Julia package for Quasi-Monte Carlo integration
  `using QMC` imports all types and functions.



## Important Notes

### IID vs Low-Discrepancy (LD) Sequences

Low-discrepancy (LD) sequences such as lattice rules are not independent like IID (independent identically distributed) points.

The code below generates 4 lattice samples of 2 dimensions and compares them to 4 IID samples.

In [3]:
println("="^60)
println("IID vs Low-Discrepancy Sequences")
println("="^60)

dd_lat = Lattice(2; randomize=true, seed=7)
x_lat = gen_samples(dd_lat, 4)
println("  4 lattice points in 2D:")
for i in 1:4
    @printf("    [%.4f  %.4f]\n", x_lat[i,1], x_lat[i,2])
end
println()

dd_iid = IIDStdUniform(2; seed=7)
x_iid = gen_samples(dd_iid, 4)
println("  4 IID uniform points in 2D:")
for i in 1:4
    @printf("    [%.4f  %.4f]\n", x_iid[i,1], x_iid[i,2])
end
println("  LD points fill the space more evenly than IID.")
println()

IID vs Low-Discrepancy Sequences
  4 lattice points in 2D:


    [0.7177  0.2410]
    [0.2177  0.7410]
    [0.9677  0.9910]
    [0.4677  0.4910]

  4 IID uniform points in 2D:
    [0.6011  0.5536]
    [0.9434  0.6319]
    [0.7177  0.0995]
    [0.2410  0.3578]
  LD points fill the space more evenly than IID.



### Multi-Dimensional Inputs

Suppose we want to create an integrand in QMC.jl for evaluating the following integral:

$$\int_{[0,1]^d} \|x\|_2^{\|x\|_2^{1/2}} \, dx,$$

where $[0,1]^d$ is the unit hypercube in $\mathbb{R}^d$.

The key point is the same as in QMCPy: the function should be able to take a set of $n$ sampling points as rows of an $n \times d$ array and return one value for each row.

In [4]:
println("="^60)
println("Custom Functions: ∫[0,1]^d ‖x‖^√(‖x‖) dx")
println("="^60)

Custom Functions: ∫[0,1]^d ‖x‖^√(‖x‖) dx


## Define a custom integrand

Define a custom integrand that works with `n × d` matrices.

Here `x` is an `n × d` matrix; we compute $\|x\|_2$ for each row and then return $\|x\|_2^{\sqrt{\|x\|_2}}$.

In [5]:
function myfunc(x)
    # x is an n × d matrix; compute ‖x‖ for each row
    x_norms = sqrt.(sum(x .^ 2; dims=2))[:]
    # Avoid 0^0: replace zeros with a small value
    x_norms[x_norms .== 0.0] .= eps()
    return x_norms .^ sqrt.(x_norms)
end

myfunc (generic function with 1 method)

Our first numerical examples use `d = 1`.

In [6]:
println("  d = 1:")
dd = IIDStdUniform(1; seed=7)
tm = Uniform(dd)
f = CustomFun(tm, myfunc)
sc = CubMCCLT(f; abs_tol=0.05)
result = integrate(sc)
true_sol_1d = 0.658582  # Wolfram: ∫₀¹ x^√x dx
@printf("    QMC: %.6f, Exact: %.6f, Error: %.2e\n",
        result.solution, true_sol_1d, abs(result.solution - true_sol_1d))
println()

  d = 1:
    QMC: 0.658965, Exact: 0.658582, Error: 3.83e-04



d = 2

In [7]:
println("  d = 2:")
dd = IIDStdUniform(2; seed=7)
tm = Uniform(dd)
f = CustomFun(tm, myfunc)
sc = CubMCCLT(f; abs_tol=0.05)
result = integrate(sc)
true_sol_2d = 0.827606  # Wolfram: ∫₀¹∫₀¹ √(x²+y²)^√(√(x²+y²)) dxdy
@printf("    QMC: %.6f, Exact: %.6f, Error: %.2e\n",
        result.solution, true_sol_2d, abs(result.solution - true_sol_2d))
println()

  d = 2:
    QMC: 0.824157, Exact: 0.827606, Error: 3.45e-03



The Four Building Blocks

In [8]:
println("="^60)
println("The Four Building Blocks")
println("="^60)

println("""
  1. Discrete Distribution — generates low-discrepancy or IID points
     Available: IIDStdUniform, Lattice, DigitalNetB2, Halton

  2. True Measure — transforms [0,1)^d points to the integration domain
     Available: Uniform, Gaussian, BrownianMotion, Lebesgue,
                GeometricBrownianMotion, StudentT, Triangular,
                Kumaraswamy, JohnsonsSU, BernoulliCont

  3. Integrand — the function to integrate
     Available: CustomFun, Keister, Genz, AsianOption, FinancialOption,
                BoxIntegral, Linear0, Sin1D, Ishigami, Hartmann6D,
                Multimodal2D, FourBranch2D

  4. Stopping Criterion — decides when the estimate is accurate enough
     Available: CubMCCLT, CubQMCLatticeG, CubQMCNetG,
                CubQMCBayesLatticeG, CubQMCBayesNetG
""")

The Four Building Blocks
  1. Discrete Distribution — generates low-discrepancy or IID points
     Available: IIDStdUniform, Lattice, DigitalNetB2, Halton

  2. True Measure — transforms [0,1)^d points to the integration domain
     Available: Uniform, Gaussian, BrownianMotion, Lebesgue,
                GeometricBrownianMotion, StudentT, Triangular,
                Kumaraswamy, JohnsonsSU, BernoulliCont

  3. Integrand — the function to integrate
     Available: CustomFun, Keister, Genz, AsianOption, FinancialOption,
                BoxIntegral, Linear0, Sin1D, Ishigami, Hartmann6D,
                Multimodal2D, FourBranch2D

  4. Stopping Criterion — decides when the estimate is accurate enough
     Available: CubMCCLT, CubQMCLatticeG, CubQMCNetG,
                CubQMCBayesLatticeG, CubQMCBayesNetG



Putting It All Together

In [9]:
println("="^60)
println("Putting It All Together: Genz Continuous Function")
println("="^60)

d = 5
a = ones(d)          # difficulty parameters
u = fill(0.5, d)     # shift parameters

Putting It All Together: Genz Continuous Function


5-element Vector{Float64}:
 0.5
 0.5
 0.5
 0.5
 0.5

IID MC

In [10]:
dd = IIDStdUniform(d; seed=7)
tm = Uniform(dd)
f = Genz(tm; kind=:continuous, a=a, u=u)
exact = genz_exact(f)
sc = CubMCCLT(f; abs_tol=0.05)
result = integrate(sc)
@printf("  IID MC:        %.6f (exact = %.6f, n = %d)\n",
        result.solution, exact, result.data[:n])

  IID MC:        0.302113 (exact = 0.301790, n = 2048)


Lattice QMC

In [11]:
dd = Lattice(d; randomize=true, seed=7)
tm = Uniform(dd)
f = Genz(tm; kind=:continuous, a=a, u=u)
sc = CubQMCLatticeG(f; abs_tol=0.001, n_init=2^8, n_reps=16)
result = integrate(sc)
@printf("  Lattice QMC:   %.6f (exact = %.6f, n/rep = %d)\n",
        result.solution, exact, result.data[:n])

  Lattice QMC:   0.301770 (exact = 0.301790, n/rep = 256)


Sobol' QMC

In [12]:
dd = DigitalNetB2(d; seed=7, randomize="LMS_DS", graycode=false)
tm = Uniform(dd)
f = Genz(tm; kind=:continuous, a=a, u=u)
sc = CubQMCNetG(f; abs_tol=0.001, n_init=2^8)
result = integrate(sc)
@printf("  Sobol' QMC:    %.6f (exact = %.6f, n = %d)\n",
        result.solution, exact, result.data[:n])
println()

println("="^60)
println("QMC.jl intro completed!")

  Sobol' QMC:    0.301810 (exact = 0.301790, n = 512)



QMC.jl intro completed!
